# Aula 23 - Notebook: Protocolo de Árvore Geradora (STP / RSTP) na Rede Ethernet do SCADA-Core

**Projeto:** SCADA-Core Automática — GRUPO 4
**Unidade Fabril:** Linha Automatizada de Seleção, Pesagem Dinâmica e Classificação Óptica de Grãos com Separação Pneumática Tripla
**Referência Normativa:** IEEE 802.1D (STP), IEEE 802.1w (RSTP), IEC 61158

Este notebook implementa, apenas com a biblioteca padrão do Python, o comportamento convergente do **RSTP** sobre a rede de 6 switches da planta descrita no arquivo `.md` desta aula:

1. Modelagem da rede como grafo ponderado $G = (V, E, w)$ com custos IEEE 802.1w;
2. Convergência **distribuída** por rodadas de troca de BPDUs (vetor de prioridade comparado em ordem lexicográfica);
3. Papéis de porta (RP / DP / AP) e estados (*Forwarding* / *Discarding*);
4. Simulação da inundação de broadcast com e sem ciclos (por que é preciso bloquear);
5. Comparação entre a árvore do RSTP e a MST de Kruskal (Aula 19);
6. Falha da fibra da câmera ($e_1$), análise $N-1$ de todos os enlaces e falha do Root Bridge;
7. Impacto do tempo de reconvergência (STP × RSTP) no permissivo `c_PERM`;
8. Asserções formais de engenharia.

### 1. Importações e Formatação Tabular em ASCII Puro

In [1]:
from collections import deque
from dataclasses import dataclass
from math import inf
from heapq import heappop, heappush


def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela vazia"
    colunas = list(dados[0].keys())
    larguras = {c: max(len(str(c)), *(len(str(linha[c])) for linha in dados)) for c in colunas}
    separador = "+" + "+".join("-" * (larguras[c] + 2) for c in colunas) + "+"
    linhas = [separador,
              "| " + " | ".join(str(c).ljust(larguras[c]) for c in colunas) + " |",
              separador]
    for linha in dados:
        linhas.append("| " + " | ".join(str(linha[c]).ljust(larguras[c]) for c in colunas) + " |")
    linhas.append(separador)
    return "\n".join(linhas)


def fmt(valor):
    """Custo com separador de milhar brasileiro."""
    return "inf" if valor == inf else f"{valor:,}".replace(",", ".")


print("Ambiente pronto (somente biblioteca padrao).")

Ambiente pronto (somente biblioteca padrao).


### 2. Classes `Switch`, `Enlace` e `RedeL2`

* O **Bridge ID** é o par `(prioridade, MAC)`. Tuplas do Python já se comparam em **ordem lexicográfica**, exatamente como a norma exige.
* O **custo** de um enlace vem da tabela IEEE 802.1w pela velocidade. A norma também permite configurar o custo da porta manualmente (`custo_manual`), o que usaremos apenas no contraexemplo da Seção 7.

In [2]:
CUSTO_IEEE = {"10 Gbps": 2_000, "1 Gbps": 20_000, "100 Mbps": 200_000}
BANDA_MBPS = {"10 Gbps": 10_000, "1 Gbps": 1_000, "100 Mbps": 100}


@dataclass(frozen=True)
class Switch:
    nome: str
    zona: str
    prioridade: int
    mac: str

    @property
    def bid(self):
        return (self.prioridade, self.mac)


@dataclass(frozen=True)
class Enlace:
    id: str
    a: str
    b: str
    velocidade: str
    custo_manual: int = 0

    @property
    def custo(self):
        return self.custo_manual or CUSTO_IEEE[self.velocidade]

    def outro(self, v):
        return self.b if v == self.a else self.a


class RedeL2:
    """Grafo nao-direcionado e ponderado dos switches de camada 2."""

    def __init__(self, nome):
        self.nome = nome
        self.switches = {}
        self.enlaces = {}

    def adicionar_switch(self, sw):
        self.switches[sw.nome] = sw

    def adicionar_enlace(self, enlace):
        self.enlaces[enlace.id] = enlace

    def bid(self, v):
        return self.switches[v].bid

    def vizinhos(self, v):
        return [(e.outro(v), e) for e in self.enlaces.values() if v in (e.a, e.b)]

    def copia(self, nome=None):
        r = RedeL2(nome or self.nome)
        r.switches = dict(self.switches)
        r.enlaces = dict(self.enlaces)
        return r

    def sem_enlace(self, id_enlace):
        r = self.copia(f"{self.nome} sem {id_enlace}")
        del r.enlaces[id_enlace]
        return r

    def sem_switch(self, nome):
        r = self.copia(f"{self.nome} sem {nome}")
        del r.switches[nome]
        r.enlaces = {k: e for k, e in r.enlaces.items() if nome not in (e.a, e.b)}
        return r

    def somente(self, ids_enlaces):
        r = self.copia(f"{self.nome} (subgrafo)")
        r.enlaces = {k: e for k, e in r.enlaces.items() if k in ids_enlaces}
        return r

    def conexo(self):
        if not self.switches:
            return True
        inicio = next(iter(self.switches))
        vistos, fila = {inicio}, deque([inicio])
        while fila:
            u = fila.popleft()
            for v, _ in self.vizinhos(u):
                if v not in vistos:
                    vistos.add(v)
                    fila.append(v)
        return len(vistos) == len(self.switches)

    def numero_ciclomatico(self):
        """|E| - |V| + 1 para grafo conexo: quantidade de ciclos independentes."""
        return len(self.enlaces) - len(self.switches) + 1


print("Classes definidas.")

Classes definidas.


### 3. Rede da Planta: 6 Switches e 7 Enlaces

Os 11 nós de automação da Aula 19 ligam-se por portas de borda ao switch da sua zona. O grafo físico é o anel `SW-CORE` → `SW-REC` → `SW-TRANS` → `SW-VIS` → `SW-EJE` → `SW-SIL` → `SW-CORE`, mais a corda de fibra $e_1$ que leva a câmera direto ao painel.

In [3]:
rede = RedeL2("Rede Ethernet SCADA-Core (Grupo 4)")
for sw in [
    Switch("SW-CORE",  "Painel central (CLP-01, IHM, SEG-901)",        4096, "00:1A:2B:00:00:01"),
    Switch("SW-REC",   "Recepcao e dosagem (FN-101, ALIM-101)",       32768, "00:1A:2B:00:00:02"),
    Switch("SW-TRANS", "Transporte e pesagem (EST-201, BAL-301)",     32768, "00:1A:2B:00:00:03"),
    Switch("SW-VIS",   "Classificacao optica (VIS-401)",               8192, "00:1A:2B:00:00:04"),
    Switch("SW-EJE",   "Ejecao e pneumatica (EJE-602/603, COMP-601)", 32768, "00:1A:2B:00:00:05"),
    Switch("SW-SIL",   "Silos (SILOS-700)",                           32768, "00:1A:2B:00:00:06"),
]:
    rede.adicionar_switch(sw)

for enlace in [
    Enlace("e1", "SW-CORE", "SW-VIS",   "1 Gbps"),
    Enlace("e2", "SW-VIS",  "SW-TRANS", "1 Gbps"),
    Enlace("e3", "SW-VIS",  "SW-EJE",   "1 Gbps"),
    Enlace("e4", "SW-EJE",  "SW-SIL",   "1 Gbps"),
    Enlace("e5", "SW-CORE", "SW-REC",   "100 Mbps"),
    Enlace("e6", "SW-REC",  "SW-TRANS", "100 Mbps"),
    Enlace("e7", "SW-SIL",  "SW-CORE",  "100 Mbps"),
]:
    rede.adicionar_enlace(enlace)

print(formatar_tabela([{"Switch": s.nome, "Zona": s.zona, "Prioridade": s.prioridade, "MAC": s.mac}
                       for s in rede.switches.values()]))
print()
print(formatar_tabela([{"Enlace": e.id, "Extremidades": f"{e.a} -- {e.b}", "Velocidade": e.velocidade, "Custo w(e)": fmt(e.custo)}
                       for e in rede.enlaces.values()]))
print(f"\n|V| = {len(rede.switches)} | |E| = {len(rede.enlaces)} | conexo? {rede.conexo()} "
      f"| numero ciclomatico = {rede.numero_ciclomatico()} (enlaces que precisarao ser bloqueados)")

+----------+---------------------------------------------+------------+-------------------+
| Switch   | Zona                                        | Prioridade | MAC               |
+----------+---------------------------------------------+------------+-------------------+
| SW-CORE  | Painel central (CLP-01, IHM, SEG-901)       | 4096       | 00:1A:2B:00:00:01 |
| SW-REC   | Recepcao e dosagem (FN-101, ALIM-101)       | 32768      | 00:1A:2B:00:00:02 |
| SW-TRANS | Transporte e pesagem (EST-201, BAL-301)     | 32768      | 00:1A:2B:00:00:03 |
| SW-VIS   | Classificacao optica (VIS-401)              | 8192       | 00:1A:2B:00:00:04 |
| SW-EJE   | Ejecao e pneumatica (EJE-602/603, COMP-601) | 32768      | 00:1A:2B:00:00:05 |
| SW-SIL   | Silos (SILOS-700)                           | 32768      | 00:1A:2B:00:00:06 |
+----------+---------------------------------------------+------------+-------------------+

+--------+--------------------+------------+------------+
| Enlace | Extremidad

### 4. Algoritmo RSTP Distribuído

Cada switch conhece apenas o que os vizinhos anunciam. Em cada **rodada**, todo switch envia em todas as portas o BPDU:

$$\big(\text{Root BID},\ \text{Root Path Cost},\ \text{BID do remetente}\big)$$

Quem recebe soma o custo do enlace de chegada e fica com o **menor vetor** entre o seu próprio (`(seu BID, 0, seu BID)`, isto é, "eu sou a raiz") e todos os recebidos, acrescentando o identificador da porta como último critério de desempate. As rodadas se repetem até que nenhum switch mude de opinião. Isso é um Bellman-Ford distribuído; o resultado é conferido depois contra o Dijkstra centralizado da Aula 14.

Com os custos convergidos, cada enlace recebe os papéis:
* **Designated Port (DP):** extremidade que anuncia o menor `(custo, BID)` no segmento;
* **Root Port (RP):** a porta por onde o switch chegou ao seu melhor vetor;
* **Alternate Port (AP):** extremidade que não é DP nem RP, ou seja, que fecharia um ciclo. Fica em *Discarding*.

In [4]:
@dataclass
class ResultadoRSTP:
    raiz: str
    custo: dict          # switch -> Root Path Cost d(v)
    porta_raiz: dict     # switch -> id do enlace da Root Port (None na raiz)
    pai: dict            # switch -> vizinho do outro lado da Root Port
    papeis: dict         # (id_enlace, switch) -> "RP" | "DP" | "AP"
    estado: dict         # id_enlace -> "Forwarding" | "Discarding"
    rodadas: int

    @property
    def ativos(self):
        return sorted(k for k, s in self.estado.items() if s == "Forwarding")

    @property
    def bloqueados(self):
        return sorted(k for k, s in self.estado.items() if s == "Discarding")


def executar_rstp(rede):
    """Simula a convergencia do RSTP (IEEE 802.1w) por rodadas sincronas de troca de BPDUs."""
    if not rede.conexo():
        raise ValueError(f"Rede particionada: {rede.nome}")

    proprio = {v: (rede.bid(v), 0, rede.bid(v), "") for v in rede.switches}
    melhor = dict(proprio)
    rodadas = 0
    while True:
        rodadas += 1
        bpdu = {v: (melhor[v][0], melhor[v][1], rede.bid(v)) for v in rede.switches}
        novo = {}
        for v in rede.switches:
            candidatos = [proprio[v]]
            for u, e in rede.vizinhos(v):
                raiz_u, custo_u, bid_u = bpdu[u]
                candidatos.append((raiz_u, custo_u + e.custo, bid_u, e.id))
            novo[v] = min(candidatos)
        if novo == melhor:
            break
        melhor = novo

    raiz = min(rede.switches, key=rede.bid)
    custo = {v: melhor[v][1] for v in rede.switches}
    porta_raiz = {v: (melhor[v][3] or None) for v in rede.switches}
    pai = {v: (rede.enlaces[p].outro(v) if p else None) for v, p in porta_raiz.items()}

    papeis, estado = {}, {}
    for e in rede.enlaces.values():
        anuncio_a = (custo[e.a], rede.bid(e.a))
        anuncio_b = (custo[e.b], rede.bid(e.b))
        designada, outra = (e.a, e.b) if anuncio_a < anuncio_b else (e.b, e.a)
        papeis[(e.id, designada)] = "DP"
        papeis[(e.id, outra)] = "RP" if porta_raiz[outra] == e.id else "AP"
        estado[e.id] = "Forwarding" if papeis[(e.id, outra)] == "RP" else "Discarding"

    return ResultadoRSTP(raiz, custo, porta_raiz, pai, papeis, estado, rodadas)


def dijkstra_custos(rede, origem):
    """Dijkstra centralizado (Aula 14), usado apenas para conferir o RSTP distribuido."""
    dist = {v: inf for v in rede.switches}
    dist[origem] = 0
    fila = [(0, origem)]
    while fila:
        d, u = heappop(fila)
        if d > dist[u]:
            continue
        for v, e in rede.vizinhos(u):
            if d + e.custo < dist[v]:
                dist[v] = d + e.custo
                heappush(fila, (dist[v], v))
    return dist


def tabela_custos(rede, res):
    linhas = []
    for v in sorted(rede.switches, key=lambda n: (res.custo[n], n)):
        linhas.append({
            "Switch": v,
            "BID": f"{rede.bid(v)[0]}/{rede.bid(v)[1][-5:]}",
            "d(v)": fmt(res.custo[v]),
            "Root Port": f"{res.porta_raiz[v]} -> {res.pai[v]}" if res.porta_raiz[v] else "- (Root Bridge)",
        })
    return formatar_tabela(linhas)


def tabela_papeis(rede, res):
    return formatar_tabela([{
        "Enlace": e.id,
        "Extremidade A": f"{e.a}: {res.papeis[(e.id, e.a)]}",
        "Extremidade B": f"{e.b}: {res.papeis[(e.id, e.b)]}",
        "Estado": res.estado[e.id],
    } for e in rede.enlaces.values()])


def imprimir_arvore(rede, res):
    filhos = {v: [] for v in rede.switches}
    for v, p in res.pai.items():
        if p:
            filhos[p].append(v)

    def visitar(v, prefixo):
        ordem = sorted(filhos[v])
        for i, f in enumerate(ordem):
            ultimo = i == len(ordem) - 1
            e = rede.enlaces[res.porta_raiz[f]]
            print(f"{prefixo}{'`-- ' if ultimo else '|-- '}[{e.id} {e.velocidade}] {f} (d = {fmt(res.custo[f])})")
            visitar(f, prefixo + ("    " if ultimo else "|   "))

    print(f"{res.raiz} (Root Bridge, d = 0)")
    visitar(res.raiz, "")


print("executar_rstp() e auxiliares definidos.")

executar_rstp() e auxiliares definidos.


### 5. Operação Normal: Convergência da Rede

In [5]:
res = executar_rstp(rede)
print(f">>> Root Bridge eleito: {res.raiz} | BID = {rede.bid(res.raiz)} | convergiu em {res.rodadas} rodadas de BPDU\n")
print(tabela_custos(rede, res))
print()
print(tabela_papeis(rede, res))
print(f"\nEnlaces ativos ({len(res.ativos)}): {res.ativos}")
print(f"Enlaces bloqueados ({len(res.bloqueados)}): {res.bloqueados}")

custos_dijkstra = dijkstra_custos(rede, res.raiz)
print(f"\nRSTP distribuido == Dijkstra centralizado? {res.custo == custos_dijkstra}")
print("\nArvore logica ativa:")
imprimir_arvore(rede, res)

>>> Root Bridge eleito: SW-CORE | BID = (4096, '00:1A:2B:00:00:01') | convergiu em 4 rodadas de BPDU

+----------+-------------+---------+-----------------+
| Switch   | BID         | d(v)    | Root Port       |
+----------+-------------+---------+-----------------+
| SW-CORE  | 4096/00:01  | 0       | - (Root Bridge) |
| SW-VIS   | 8192/00:04  | 20.000  | e1 -> SW-CORE   |
| SW-EJE   | 32768/00:05 | 40.000  | e3 -> SW-VIS    |
| SW-TRANS | 32768/00:03 | 40.000  | e2 -> SW-VIS    |
| SW-SIL   | 32768/00:06 | 60.000  | e4 -> SW-EJE    |
| SW-REC   | 32768/00:02 | 200.000 | e5 -> SW-CORE   |
+----------+-------------+---------+-----------------+

+--------+---------------+---------------+------------+
| Enlace | Extremidade A | Extremidade B | Estado     |
+--------+---------------+---------------+------------+
| e1     | SW-CORE: DP   | SW-VIS: RP    | Forwarding |
| e2     | SW-VIS: DP    | SW-TRANS: RP  | Forwarding |
| e3     | SW-VIS: DP    | SW-EJE: RP    | Forwarding |
| e4     | 

**Leitura:** o `SW-SIL` tem cabo direto para o painel ($e_7$), mas chega à raiz por três saltos de fibra ($60.000$) em vez de um salto de cobre ($200.000$). Por isso a sua extremidade de $e_7$ vira *Alternate Port*. O mesmo acontece com o `SW-REC` em $e_6$: o `SW-TRANS` anuncia custo $40.000$, menor que os $200.000$ do `SW-REC`, e fica com a porta designada.

### 6. Por Que Bloquear: Inundação de Broadcast sem TTL

Simulamos um quadro de broadcast (ex.: uma requisição ARP da balança `BAL-301`) emitido pelo `SW-TRANS`. Cada switch repete o quadro em todas as portas ativas, menos a de chegada, e nenhum quadro é descartado por idade, porque o cabeçalho Ethernet não tem TTL. Contamos quantas cópias estão trafegando a cada salto:
* com **todos os 7 enlaces** encaminhando (sem STP);
* apenas com os **5 enlaces da árvore** do RSTP.

In [6]:
def simular_broadcast(rede, origem, enlaces_ativos, saltos):
    """Inundacao sem TTL: retorna o numero de copias em transito apos cada salto."""
    quadros = [(origem, None)]  # (switch onde o quadro esta, enlace por onde chegou)
    historico = []
    for _ in range(saltos):
        novos = []
        for v, chegada in quadros:
            for viz, e in rede.vizinhos(v):
                if e.id in enlaces_ativos and e.id != chegada:
                    novos.append((viz, e.id))
        quadros = novos
        historico.append(len(quadros))
    return historico


SALTOS = 12
sem_stp = simular_broadcast(rede, "SW-TRANS", set(rede.enlaces), SALTOS)
com_rstp = simular_broadcast(rede, "SW-TRANS", set(res.ativos), SALTOS)
print(formatar_tabela([{"Salto": i + 1, "Copias sem STP (7 enlaces)": a, "Copias com RSTP (5 enlaces)": b}
                       for i, (a, b) in enumerate(zip(sem_stp, com_rstp))]))
print(f"\nSem STP: {sum(sem_stp)} copias acumuladas em {SALTOS} saltos e o quadro nunca para de circular.")
print(f"Com RSTP: o quadro alcanca todos os switches e se extingue (ultimo salto com copias: "
      f"{max(i + 1 for i, n in enumerate(com_rstp) if n > 0)}).")

+-------+----------------------------+-----------------------------+
| Salto | Copias sem STP (7 enlaces) | Copias com RSTP (5 enlaces) |
+-------+----------------------------+-----------------------------+
| 1     | 2                          | 1                           |
| 2     | 3                          | 2                           |
| 3     | 5                          | 2                           |
| 4     | 6                          | 0                           |
| 5     | 7                          | 0                           |
| 6     | 11                         | 0                           |
| 7     | 16                         | 0                           |
| 8     | 19                         | 0                           |
| 9     | 25                         | 0                           |
| 10    | 38                         | 0                           |
| 11    | 51                         | 0                           |
| 12    | 63                      

### 7. Árvore do RSTP × Árvore Geradora Mínima (Kruskal)

A MST minimiza a **soma** dos pesos e não depende de raiz; a árvore do RSTP minimiza a **distância de cada switch à raiz** (árvore de caminhos mínimos). Primeiro, o contraexemplo mínimo de 3 switches, com custos de porta configurados manualmente:

In [7]:
def kruskal(rede):
    """MST por Kruskal com Union-Find (Aula 19). Empates mantem a ordem dos ids."""
    pai = {v: v for v in rede.switches}

    def achar(x):
        while pai[x] != x:
            pai[x] = pai[pai[x]]
            x = pai[x]
        return x

    arvore = []
    for e in sorted(rede.enlaces.values(), key=lambda e: e.custo):
        ra, rb = achar(e.a), achar(e.b)
        if ra != rb:
            pai[ra] = rb
            arvore.append(e.id)
    return sorted(arvore)


def peso(rede, ids):
    return sum(rede.enlaces[i].custo for i in ids)


def distancias_na_arvore(rede, ids, raiz):
    """Custo acumulado da raiz ate cada switch usando apenas as arestas da arvore."""
    sub = rede.somente(set(ids))
    dist, fila = {raiz: 0}, deque([raiz])
    while fila:
        u = fila.popleft()
        for v, e in sub.vizinhos(u):
            if v not in dist:
                dist[v] = dist[u] + e.custo
                fila.append(v)
    return dist


triangulo = RedeL2("Contraexemplo")
for nome, prio, mac in [("R", 4096, "00:00:00:00:00:01"), ("A", 32768, "00:00:00:00:00:02"), ("B", 32768, "00:00:00:00:00:03")]:
    triangulo.adicionar_switch(Switch(nome, "-", prio, mac))
triangulo.adicionar_enlace(Enlace("R-A", "R", "A", "1 Gbps", custo_manual=2))
triangulo.adicionar_enlace(Enlace("A-B", "A", "B", "1 Gbps", custo_manual=2))
triangulo.adicionar_enlace(Enlace("R-B", "R", "B", "1 Gbps", custo_manual=3))

mst_tri = kruskal(triangulo)
rstp_tri = executar_rstp(triangulo)
d_mst_tri = distancias_na_arvore(triangulo, mst_tri, "R")
print(formatar_tabela([
    {"Arvore": "MST (Kruskal)", "Arestas": mst_tri, "Peso total": peso(triangulo, mst_tri), "d(B)": d_mst_tri["B"]},
    {"Arvore": "RSTP (raiz R)", "Arestas": rstp_tri.ativos, "Peso total": peso(triangulo, rstp_tri.ativos), "d(B)": rstp_tri.custo["B"]},
]))

+---------------+----------------+------------+------+
| Arvore        | Arestas        | Peso total | d(B) |
+---------------+----------------+------------+------+
| MST (Kruskal) | ['A-B', 'R-A'] | 4          | 4    |
| RSTP (raiz R) | ['R-A', 'R-B'] | 5          | 3    |
+---------------+----------------+------------+------+


#### 7.1. Na rede da planta

Em operação normal, a árvore do RSTP coincide com uma das MSTs. Após o rompimento de $e_1$, as duas árvores têm o **mesmo peso total**, mas são diferentes. A diferença aparece na distância da câmera (`SW-VIS`) até o painel:

In [8]:
linhas = []
for rotulo, r in [("Operacao normal", rede), ("Apos rompimento de e1", rede.sem_enlace("e1"))]:
    mst = kruskal(r)
    arvore_rstp = executar_rstp(r)
    d_mst = distancias_na_arvore(r, mst, "SW-CORE")
    linhas.append({"Cenario": rotulo, "Arvore": "MST (Kruskal)", "Arestas": " ".join(mst),
                   "Peso": fmt(peso(r, mst)), "d(SW-VIS)": fmt(d_mst["SW-VIS"])})
    linhas.append({"Cenario": rotulo, "Arvore": "RSTP", "Arestas": " ".join(arvore_rstp.ativos),
                   "Peso": fmt(peso(r, arvore_rstp.ativos)), "d(SW-VIS)": fmt(arvore_rstp.custo["SW-VIS"])})
print(formatar_tabela(linhas))

+-----------------------+---------------+----------------+---------+-----------+
| Cenario               | Arvore        | Arestas        | Peso    | d(SW-VIS) |
+-----------------------+---------------+----------------+---------+-----------+
| Operacao normal       | MST (Kruskal) | e1 e2 e3 e4 e5 | 280.000 | 20.000    |
| Operacao normal       | RSTP          | e1 e2 e3 e4 e5 | 280.000 | 20.000    |
| Apos rompimento de e1 | MST (Kruskal) | e2 e3 e4 e5 e6 | 460.000 | 420.000   |
| Apos rompimento de e1 | RSTP          | e2 e3 e4 e5 e7 | 460.000 | 240.000   |
+-----------------------+---------------+----------------+---------+-----------+


### 8. Falha Crítica: Rompimento da Fibra da Câmera ($e_1$)

Removemos $e_1$ do grafo físico e deixamos a rede reconvergir. Comparamos o papel de cada porta antes e depois, e o **gargalo de banda** no caminho da câmera até o painel (menor velocidade entre os enlaces desse caminho na árvore).

In [9]:
def caminho_ate_raiz(rede, res, origem):
    caminho, v = [], origem
    while res.pai[v]:
        caminho.append(rede.enlaces[res.porta_raiz[v]])
        v = res.pai[v]
    return caminho


def gargalo_mbps(rede, res, origem):
    return min(BANDA_MBPS[e.velocidade] for e in caminho_ate_raiz(rede, res, origem))


rede_e1 = rede.sem_enlace("e1")
res_e1 = executar_rstp(rede_e1)
print(f">>> Rede sem e1: conexa? {rede_e1.conexo()} | Root Bridge: {res_e1.raiz} | {res_e1.rodadas} rodadas\n")
print(tabela_custos(rede_e1, res_e1))

transicoes = []
for e in rede.enlaces.values():
    for v in (e.a, e.b):
        antes = res.papeis[(e.id, v)]
        depois = res_e1.papeis.get((e.id, v), "rompido")
        if antes != depois:
            transicoes.append({"Enlace": e.id, "Porta em": v, "Antes": antes, "Depois": depois})
print("\nPortas que mudaram de papel:")
print(formatar_tabela(transicoes))
print(f"\nBloqueados antes: {res.bloqueados} | bloqueados depois: {res_e1.bloqueados}")

for rotulo, r, rr in [("Normal", rede, res), ("Sem e1", rede_e1, res_e1)]:
    caminho = " -> ".join(f"{e.id}({e.velocidade})" for e in caminho_ate_raiz(r, rr, "SW-VIS"))
    print(f"{rotulo:7s} | SW-VIS ate SW-CORE: {caminho} | gargalo = {gargalo_mbps(r, rr, 'SW-VIS')} Mbps")

print("\nArvore logica apos a falha:")
imprimir_arvore(rede_e1, res_e1)

>>> Rede sem e1: conexa? True | Root Bridge: SW-CORE | 5 rodadas

+----------+-------------+---------+-----------------+
| Switch   | BID         | d(v)    | Root Port       |
+----------+-------------+---------+-----------------+
| SW-CORE  | 4096/00:01  | 0       | - (Root Bridge) |
| SW-REC   | 32768/00:02 | 200.000 | e5 -> SW-CORE   |
| SW-SIL   | 32768/00:06 | 200.000 | e7 -> SW-CORE   |
| SW-EJE   | 32768/00:05 | 220.000 | e4 -> SW-SIL    |
| SW-VIS   | 8192/00:04  | 240.000 | e3 -> SW-EJE    |
| SW-TRANS | 32768/00:03 | 260.000 | e2 -> SW-VIS    |
+----------+-------------+---------+-----------------+

Portas que mudaram de papel:
+--------+----------+-------+---------+
| Enlace | Porta em | Antes | Depois  |
+--------+----------+-------+---------+
| e1     | SW-CORE  | DP    | rompido |
| e1     | SW-VIS   | RP    | rompido |
| e3     | SW-VIS   | DP    | RP      |
| e3     | SW-EJE   | RP    | DP      |
| e4     | SW-EJE   | DP    | RP      |
| e4     | SW-SIL   | RP    | DP  

**Modo degradado:** nenhum switch fica isolado, mas as imagens do `VIS-401` passam a subir para o painel por um enlace de $100\text{ Mbps}$. O SCADA deve reduzir o streaming de imagens e alarmar a perda de redundância até o reparo da fibra.

#### 8.1. Análise $N-1$: rompimento de cada enlace

Na Aula 19, a MST tinha todas as arestas como pontes: qualquer cabo rompido isolava parte da planta. Agora repetimos o teste para cada um dos 7 enlaces da rede redundante.

In [10]:
linhas_n1 = []
for id_enlace in rede.enlaces:
    r = rede.sem_enlace(id_enlace)
    if not r.conexo():
        linhas_n1.append({"Enlace rompido": id_enlace, "Rede conexa?": "NAO", "Bloqueados": "-",
                          "d(SW-VIS)": "-", "Gargalo camera (Mbps)": "-"})
        continue
    rr = executar_rstp(r)
    linhas_n1.append({"Enlace rompido": id_enlace, "Rede conexa?": "sim", "Bloqueados": " ".join(rr.bloqueados),
                      "d(SW-VIS)": fmt(rr.custo["SW-VIS"]), "Gargalo camera (Mbps)": gargalo_mbps(r, rr, "SW-VIS")})
print(formatar_tabela(linhas_n1))
falhas_isolantes = [l["Enlace rompido"] for l in linhas_n1 if l["Rede conexa?"] == "NAO"]
print(f"\nEnlaces cuja falha isola algum switch: {falhas_isolantes or 'nenhum'}")

+----------------+--------------+------------+-----------+-----------------------+
| Enlace rompido | Rede conexa? | Bloqueados | d(SW-VIS) | Gargalo camera (Mbps) |
+----------------+--------------+------------+-----------+-----------------------+
| e1             | sim          | e6         | 240.000   | 100                   |
| e2             | sim          | e7         | 20.000    | 1000                  |
| e3             | sim          | e6         | 20.000    | 1000                  |
| e4             | sim          | e6         | 20.000    | 1000                  |
| e5             | sim          | e7         | 20.000    | 1000                  |
| e6             | sim          | e7         | 20.000    | 1000                  |
| e7             | sim          | e6         | 20.000    | 1000                  |
+----------------+--------------+------------+-----------+-----------------------+

Enlaces cuja falha isola algum switch: nenhum


#### 8.2. Falha do próprio Root Bridge (`SW-CORE`)

Sem o `SW-CORE`, o switch de menor BID restante deve assumir a raiz. Pela configuração de prioridades, esse é o `SW-VIS` (8192).

In [11]:
rede_sem_core = rede.sem_switch("SW-CORE")
res_sem_core = executar_rstp(rede_sem_core)
print(f">>> Novo Root Bridge: {res_sem_core.raiz} | BID = {rede_sem_core.bid(res_sem_core.raiz)}")
print(f"Grafo restante: |V| = {len(rede_sem_core.switches)}, |E| = {len(rede_sem_core.enlaces)}, "
      f"numero ciclomatico = {rede_sem_core.numero_ciclomatico()} -> bloqueados: {res_sem_core.bloqueados or 'nenhum'}\n")
imprimir_arvore(rede_sem_core, res_sem_core)
print("\nObservacao: a rede de campo sobrevive, mas o CLP-01 e o SCADA estavam no SW-CORE.")
print("Sem o controlador, a planta vai para o estado seguro por perda de comunicacao.")

>>> Novo Root Bridge: SW-VIS | BID = (8192, '00:1A:2B:00:00:04')
Grafo restante: |V| = 5, |E| = 4, numero ciclomatico = 0 -> bloqueados: nenhum

SW-VIS (Root Bridge, d = 0)
|-- [e3 1 Gbps] SW-EJE (d = 20.000)
|   `-- [e4 1 Gbps] SW-SIL (d = 40.000)
`-- [e2 1 Gbps] SW-TRANS (d = 20.000)
    `-- [e6 100 Mbps] SW-REC (d = 220.000)

Observacao: a rede de campo sobrevive, mas o CLP-01 e o SCADA estavam no SW-CORE.
Sem o controlador, a planta vai para o estado seguro por perda de comunicacao.


### 9. STP × RSTP: Tempo de Reconvergência e o Permissivo `c_PERM`

O permissivo da Aula 2 depende do estado da câmera:

$$c_{\text{PERM}} \leftrightarrow \neg p_{\text{EMERG}} \land \neg p_{\text{JI201}} \land \neg p_{\text{PAL601}} \land p_{\text{KSA401}} \land \neg p_{\text{SILO\_CHEIO}}$$

Se o CLP ficar sem notícias do `VIS-401` por mais tempo que o seu *timeout* de comunicação, ele assume $p_{\text{KSA401}} = 0$. Com a esteira a $2{,}0\text{ m/s}$ (Aula 10, caso C01), calculamos quantos metros de grãos passam durante a reconvergência e se a linha desarma.

In [12]:
V_ESTEIRA = 2.0     # m/s  - velocidade nominal (Aula 10, caso C01)
TIMEOUT_CLP = 0.5   # s    - timeout de comunicacao proposto para o CLP


def c_perm(p_emerg, p_ji201, p_pal601, p_ksa401, p_silo_cheio):
    return (not p_emerg) and (not p_ji201) and (not p_pal601) and p_ksa401 and (not p_silo_cheio)


cenarios = [
    ("STP 802.1D (melhor caso)", 30.0),
    ("STP 802.1D (pior caso)", 50.0),
    ("RSTP 802.1w (tipico)", 0.05),
    ("RSTP 802.1w (pior tipico)", 0.2),
]
linhas_t = []
for nome, t in cenarios:
    p_ksa401 = t <= TIMEOUT_CLP
    permissivo = c_perm(False, False, False, p_ksa401, False)
    linhas_t.append({"Protocolo": nome, "Reconvergencia (s)": t, "Esteira sem rede (m)": round(V_ESTEIRA * t, 2),
                     "p_KSA401": int(p_ksa401), "c_PERM": int(permissivo),
                     "Resultado": "alarme, linha segue" if permissivo else "TRIP da linha"})
print(formatar_tabela(linhas_t))

+---------------------------+--------------------+----------------------+----------+--------+---------------------+
| Protocolo                 | Reconvergencia (s) | Esteira sem rede (m) | p_KSA401 | c_PERM | Resultado           |
+---------------------------+--------------------+----------------------+----------+--------+---------------------+
| STP 802.1D (melhor caso)  | 30.0               | 60.0                 | 0        | 0      | TRIP da linha       |
| STP 802.1D (pior caso)    | 50.0               | 100.0                | 0        | 0      | TRIP da linha       |
| RSTP 802.1w (tipico)      | 0.05               | 0.1                  | 1        | 1      | alarme, linha segue |
| RSTP 802.1w (pior tipico) | 0.2                | 0.4                  | 1        | 1      | alarme, linha segue |
+---------------------------+--------------------+----------------------+----------+--------+---------------------+


### 10. Asserções e Validações Formais de Engenharia

In [13]:
def eh_arvore_geradora(rede, ids):
    sub = rede.somente(set(ids))
    return sub.conexo() and len(ids) == len(rede.switches) - 1


# Validacao 1: o Root Bridge eleito e o painel do CLP (menor BID configurado)
assert res.raiz == "SW-CORE", "Falha: a raiz deve ser o SW-CORE (prioridade 4096)!"

# Validacao 2: enlaces ativos formam arvore geradora (conexa, |E_T| = |V| - 1)
assert eh_arvore_geradora(rede, res.ativos), "Falha: os enlaces Forwarding nao formam arvore geradora!"

# Validacao 3: numero de bloqueados = numero ciclomatico, e sao exatamente e6 e e7
assert len(res.bloqueados) == rede.numero_ciclomatico() == 2
assert res.bloqueados == ["e6", "e7"], f"Falha: bloqueados inesperados {res.bloqueados}"

# Validacao 4: o RSTP distribuido converge para as mesmas distancias do Dijkstra centralizado
assert res.custo == dijkstra_custos(rede, res.raiz), "Falha: RSTP divergiu do Dijkstra!"
assert res.custo == {"SW-CORE": 0, "SW-VIS": 20_000, "SW-TRANS": 40_000, "SW-EJE": 40_000,
                     "SW-SIL": 60_000, "SW-REC": 200_000}

# Validacao 5: cada switch nao raiz tem exatamente uma Root Port; a raiz nao tem nenhuma
for v in rede.switches:
    rps = [k for k, p in res.papeis.items() if k[1] == v and p == "RP"]
    assert len(rps) == (0 if v == res.raiz else 1), f"Falha: {v} com {len(rps)} Root Ports!"

# Validacao 6: broadcast se extingue na arvore e circula indefinidamente com ciclos
assert com_rstp[-1] == 0 and sem_stp[-1] > 0, "Falha: comportamento de inundacao inesperado!"

# Validacao 7: apos romper e1, a AP de SW-SIL vira RP e o bloqueio de e6 passa para o SW-TRANS
assert res.papeis[("e7", "SW-SIL")] == "AP" and res_e1.papeis[("e7", "SW-SIL")] == "RP"
assert res_e1.bloqueados == ["e6"] and res_e1.papeis[("e6", "SW-TRANS")] == "AP"
assert eh_arvore_geradora(rede_e1, res_e1.ativos)
# ...e a porta de SW-VIS em e3 vira Root Port: por isso nao pode haver Root Guard nas portas de anel do SW-VIS
assert res.papeis[("e3", "SW-VIS")] == "DP" and res_e1.porta_raiz["SW-VIS"] == "e3"
# Root Guard em todas as portas do SW-CORE e seguro: a raiz nunca tem Root Port em nenhum cenario com ela ativa
assert all(r.porta_raiz["SW-CORE"] is None for r in (res, res_e1))

# Validacao 8: MST e arvore do RSTP tem o mesmo peso apos a falha, mas a do RSTP aproxima a camera do painel
mst_e1 = kruskal(rede_e1)
assert peso(rede_e1, mst_e1) == peso(rede_e1, res_e1.ativos) == 460_000
assert mst_e1 != res_e1.ativos
assert distancias_na_arvore(rede_e1, mst_e1, "SW-CORE")["SW-VIS"] == 420_000
assert res_e1.custo["SW-VIS"] == 240_000

# Validacao 9: contraexemplo - MST tem menor peso, RSTP tem menor d(B)
assert peso(triangulo, mst_tri) == 4 and peso(triangulo, rstp_tri.ativos) == 5
assert d_mst_tri["B"] == 4 and rstp_tri.custo["B"] == 3

# Validacao 10: nenhum rompimento simples de cabo isola um switch (N-1)
assert not falhas_isolantes, f"Falha: enlaces que isolam a rede {falhas_isolantes}"

# Validacao 11: sem o SW-CORE, o root secundario configurado (SW-VIS) assume
assert res_sem_core.raiz == "SW-VIS" and res_sem_core.bloqueados == []

# Validacao 12: RSTP nao desarma a linha; STP desarma
assert all(l["c_PERM"] == 1 for l in linhas_t if l["Protocolo"].startswith("RSTP"))
assert all(l["c_PERM"] == 0 for l in linhas_t if l["Protocolo"].startswith("STP"))

print("Todas as 12 validacoes formais passaram com sucesso.")

Todas as 12 validacoes formais passaram com sucesso.
